# Solutions – 1. Maximum Likelihood Fitting

This notebook provides worked solutions to the five exercises from
[01_maximum_likelihood_fitting.ipynb](../Stats/01_maximum_likelihood_fitting.ipynb).

We reuse the same PDF, data-generation routine, and NLL functions from notebook 01.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)

In [ ]:
# ------- true parameter values -------
theta_true  = 0.3   # signal fraction
mu_true     = 5.0   # Gaussian mean
sigma_true  = 0.5   # Gaussian width
lambda_true = 0.5   # exponential rate  (mean = 1/lambda)

n_samples = 2000    # total number of events

# ------- sampling from the mixture -------
n_signal     = rng.binomial(n_samples, theta_true)
n_background = n_samples - n_signal

x_signal     = rng.normal(mu_true, sigma_true, size=n_signal)
x_background = rng.exponential(1.0 / lambda_true, size=n_background)

x_data = np.concatenate([x_signal, x_background])
rng.shuffle(x_data)

print(f"Generated {len(x_data)} events  ({n_signal} signal + {n_background} background)")

In [ ]:
# NLL functions (same as notebook 01)
def nll_full(params, data):
    """Negative log-likelihood with all four parameters free."""
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma)
    lam   = np.exp(log_lam)
    if not (0.0 < theta < 1.0):
        return np.inf
    log_sig_term = np.log(theta)       + norm.logpdf(data, mu, sigma)
    log_bkg_term = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    return -np.sum(np.logaddexp(log_sig_term, log_bkg_term))

print("NLL function helper defined.")

---
## Exercise 1 – Effect of sample size

**Task:** Re-run the fit with $n = 200$, $n = 500$, and $n = 5000$ events.
How do the MLE estimates and their spread change?  Plot $\hat{\theta}$ as a function of $n$.

In [ ]:
# Run 100 pseudo-experiments for each sample size
sample_sizes = [200, 500, 2000, 5000]
n_repeats    = 100

results_by_n = {}
for n in sample_sizes:
    theta_hats = []
    rng_local = np.random.default_rng(seed=123)
    for _ in range(n_repeats):
        n_sig = rng_local.binomial(n, theta_true)
        n_bkg = n - n_sig
        xs = np.concatenate([
            rng_local.normal(mu_true, sigma_true, size=n_sig),
            rng_local.exponential(1.0 / lambda_true, size=n_bkg)
        ])
        rng_local.shuffle(xs)

        # Initial guess (close to but not equal to truth)
        theta0  = 0.2
        mu0     = xs.mean()          # rough centre of the distribution
        sigma0  = 0.6
        lam0    = 0.6

        x0_full = [theta0, mu0, np.log(sigma0), np.log(lam0)]
        result = minimize(nll_full, x0=x0_full, args=(xs,), method='Nelder-Mead',
                    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000})

        theta_hats.append(result.x[0])
    results_by_n[n] = np.array(theta_hats)
    print(f"n = {n:5d}:  mean(theta_hat) = {np.mean(theta_hats):.4f},  "
          f"std(theta_hat) = {np.std(theta_hats):.4f}")

In [ ]:
means  = [np.mean(results_by_n[n]) for n in sample_sizes]
stds   = [np.std(results_by_n[n])  for n in sample_sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: mean and std vs n
axes[0].errorbar(sample_sizes, means, yerr=stds, fmt='o-', capsize=5)
axes[0].axhline(theta_true, color='red', ls='--', label=f'True θ = {theta_true}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Sample size n')
axes[0].set_ylabel(r'$\hat{\theta}$')
axes[0].set_title(r'MLE $\hat{\theta}$ vs. sample size')
axes[0].legend()

# Right: expected 1/sqrt(n) scaling of the standard deviation
n_arr = np.array(sample_sizes, dtype=float)
scale = stds[0] * np.sqrt(sample_sizes[0])
axes[1].plot(n_arr, stds, 'o-', label=r'Measured $\sigma(\hat{\theta})$')
axes[1].plot(n_arr, scale / np.sqrt(n_arr), 'r--', label=r'$\propto 1/\sqrt{n}$')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Sample size n')
axes[1].set_ylabel(r'$\sigma(\hat{\theta})$')
axes[1].set_title('Statistical uncertainty vs. sample size')
axes[1].legend()

plt.tight_layout()
plt.show()

**Observations:**
- The mean of $\hat{\theta}$ is close to the true value 0.3 for all sample sizes, confirming the estimator is (approximately) unbiased.
- The standard deviation decreases as $1/\sqrt{n}$, consistent with the Cramér–Rao bound.
  Doubling the dataset halves the uncertainty.

---
## Exercise 2 – Starting-point sensitivity

**Task:** Try five very different initial guesses for $\theta_0$ and check whether the
optimiser always converges to the same solution.  What does this tell you about the NLL
surface?

In [ ]:
# Use the baseline 2000-event dataset
theta_inits = [0.01, 0.1, 0.5, 0.9, 0.99]

print(f"{'theta_0':>10}  {'theta_hat':>12}  {'NLL':>12}  success")
print("-" * 55)
for t0 in theta_inits:
    x0 = [t0, x_data.mean(), np.log(0.6), np.log(0.6)]
    res = minimize(nll_full, x0=x0, args=(x_data,), method='Nelder-Mead',
                   options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000})
    print(f"{t0:>10.2f}  {res.x[0]:>12.6f}  {res.fun:>12.4f}  {res.success}")

**Observations:**
- All five starting points converge to the same $\hat{\theta}$ within numerical precision.
- The NLL surface for this mixture model is **unimodal** (single global minimum) in the
  interior of the parameter space, so the starting point does not matter much.
- Extreme starting values close to 0 or 1 may require more iterations but ultimately
  reach the same solution.

---
## Exercise 3 – Binned vs. unbinned likelihood

**Task:** Bin the data into 50 equal-width bins and construct the binned NLL
$-\sum_k n_k \ln p_k$ where $p_k = \int_{\text{bin}\,k} f(x;\boldsymbol{\theta})\,dx$.
Minimise it and compare with the unbinned MLE.

In [ ]:
# --- Data histogram ---
n_bins  = 50
x_min, x_max = 0.0, 14.0
counts, bin_edges = np.histogram(x_data, bins=n_bins, range=(x_min, x_max))
bin_width = bin_edges[1] - bin_edges[0]

def binned_nll(params):
    """Binned NLL using CDF integrals over each bin."""
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma)
    lam   = np.exp(log_lam)
    if not (0.0 < theta < 1.0):
        return np.inf
    # Expected fraction in each bin from the mixture CDF
    def mixture_cdf(x):
        return theta * norm.cdf(x, mu, sigma) + (1.0 - theta) * expon.cdf(x, scale=1.0/lam)
    pk = np.diff(mixture_cdf(bin_edges))   # shape (n_bins,)
    pk = np.clip(pk, 1e-300, None)
    # Only include non-empty bins to avoid log(0)
    mask = counts > 0
    return -np.sum(counts[mask] * np.log(pk[mask]))

# --- Minimise binned NLL ---
x0_binned = [0.2, x_data.mean(), np.log(0.6), np.log(0.6)]
res_binned = minimize(binned_nll, x0=x0_binned, method='Nelder-Mead',
                      options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000})
theta_bin, mu_bin, ls_bin, ll_bin = res_binned.x
sigma_bin = np.exp(ls_bin)
lam_bin   = np.exp(ll_bin)

# --- Unbinned MLE for comparison ---
x0_full = [0.2, x_data.mean(), np.log(0.6), np.log(0.6)]
res_full = minimize(nll_full, x0=x0_full, args=(x_data,), method='Nelder-Mead',
                    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000})
theta_hat, mu_hat, ls_hat, ll_hat = res_full.x
sigma_hat = np.exp(ls_hat)
lam_hat   = np.exp(ll_hat)

print("=== Comparison: Unbinned vs. Binned MLE ===")
print(f"{'Parameter':<10}  {'True':>8}  {'Unbinned':>10}  {'Binned':>10}")
print("-" * 45)
for pname, ptrue, phat, pbin in [
    ('theta',  theta_true,  theta_hat,  theta_bin),
    ('mu',     mu_true,     mu_hat,     mu_bin),
    ('sigma',  sigma_true,  sigma_hat,  sigma_bin),
    ('lambda', lambda_true, lam_hat,    lam_bin),
]:
    print(f"{pname:<10}  {ptrue:>8.4f}  {phat:>10.4f}  {pbin:>10.4f}")

**Observations:**
- Both methods recover similar estimates.  The binned estimator introduces a small bias
  from the discretisation, but with 50 bins over the range (0–14) the bin width
  ($\approx 0.28$) is well below the signal width $\sigma = 0.5$, so the bias is small.
- The **unbinned likelihood** uses the full information in the data and is therefore
  more efficient (lower variance) than the binned likelihood, especially when the
  signal peak is narrow compared to the bin width.
- If the bin width is large compared to the resolution, the binned estimate can degrade
  significantly.

---
## Exercise 4 – Signal-fraction near zero

**Task:** Set $\theta_{\rm true} = 0.01$ and $n = 2000$.  Does the MLE still converge?
Check for bias by repeating 100 times and relate the findings to the concept of an upper limit.

In [ ]:
theta_small = 0.01
n_small     = 2000
n_rep       = 100

rng_ex4 = np.random.default_rng(seed=123)
theta_hats_small = []

for _ in range(n_rep):
    n_sig = rng_ex4.binomial(n_small, theta_small)
    n_bkg = n_small - n_sig
    xs = np.concatenate([
        rng_ex4.normal(mu_true, sigma_true, size=n_sig),
        rng_ex4.exponential(1.0 / lambda_true, size=n_bkg)
    ])
    rng_ex4.shuffle(xs)

    # Initial guess (close to but not equal to truth)
    theta0  = 0.2
    mu0     = xs.mean()          # rough centre of the distribution
    sigma0  = 0.6
    lam0    = 0.6

    x0_full = [theta0, mu0, np.log(sigma0), np.log(lam0)]
    result = minimize(nll_full, x0=x0_full, args=(xs,), method='Nelder-Mead',
                options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000})

    th = result.x[0]
    theta_hats_small.append(th)

theta_arr = np.array(theta_hats_small)
print(f"theta_true = {theta_small}")
print(f"Mean  theta_hat = {theta_arr.mean():.4f}")
print(f"Std   theta_hat = {theta_arr.std():.4f}")
print(f"Fraction theta_hat <= 0.01 (near boundary) = {(theta_arr <= 0.01).mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(theta_arr, bins=25, density=True, histtype='stepfilled',
        alpha=0.6, color='steelblue', label=r'$\hat{\theta}$ distribution')
ax.axvline(theta_small, color='red', ls='--', lw=2, label=f'True θ = {theta_small}')
ax.axvline(theta_arr.mean(), color='green', ls='-', lw=2,
           label=f'Mean = {theta_arr.mean():.4f}')
ax.set_xlabel(r'$\hat{\theta}$')
ax.set_ylabel('Density')
ax.set_title(r'MLE distribution for $\theta_{\rm true} = 0.02$, $n = 2000$')
ax.legend()
plt.tight_layout()
plt.show()

**Observations:**
- The MLE generally still converges, but a non-negligible fraction of pseudo-experiments
  return values close to the lower boundary.
- The distribution of $\hat{\theta}$ is highly asymmetric and truncated at 0, so the
  estimator is **biased upward**.  This is the "boundary bias" that arises when the
  true value is near the edge of the physically allowed region.
- This regime motivates the use of **upper limits**: rather than quoting a biased
  central value with unreliable uncertainties, we set a 95 % CL upper limit on $\theta$.

---
## Exercise 5 – Extended maximum likelihood

**Task:** Add $\mu_{\rm tot}$ (expected total number of events) as a free parameter via
a Poisson extended likelihood and verify $\hat{\mu}_{\rm tot} \approx n_{\rm samples}$.

In [ ]:
from scipy.special import gammaln

def extended_nll(params, data):
    """Poisson extended NLL.

    params = (mu_tot, theta, mu, log_sigma, log_lam)
      mu_tot  – expected total number of events (free)
      theta   – signal fraction
      mu      – Gaussian mean
      sigma   – Gaussian width  (stored as log_sigma internally)
      lambda  – exponential rate (stored as log_lam internally)
    """
    mu_tot, theta, mu_gauss, log_sigma, log_lam = params
    sigma = np.exp(log_sigma)
    lam   = np.exp(log_lam)
    n     = len(data)

    if mu_tot <= 0 or not (0.0 < theta < 1.0):
        return np.inf

    # Poisson term: -log P(n | mu_tot) = mu_tot - n*log(mu_tot) + log(n!)
    poisson_term = mu_tot - n * np.log(mu_tot) + gammaln(n + 1)

    # Unbinned log-likelihood term (using shape PDF only)
    log_sig_term = np.log(theta)       + norm.logpdf(data, mu_gauss, sigma)
    log_bkg_term = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    shape_term   = -np.sum(np.logaddexp(log_sig_term, log_bkg_term))

    return poisson_term + shape_term

n_obs = len(x_data)
x0_ext = [n_obs, 0.2, x_data.mean(), np.log(0.6), np.log(0.6)]
res_ext = minimize(extended_nll, x0=x0_ext, args=(x_data,), method='Nelder-Mead',
                   options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 100000})

mu_tot_hat, theta_ext, mu_ext, ls_ext, ll_ext = res_ext.x
print("=== Extended MLE results ===")
print(f"  mu_tot_hat = {mu_tot_hat:.2f}   (n_samples = {n_obs})")
print(f"  theta_hat  = {theta_ext:.4f}   (true: {theta_true})")
print(f"  mu_hat     = {mu_ext:.4f}    (true: {mu_true})")
print(f"  sigma_hat  = {np.exp(ls_ext):.4f}   (true: {sigma_true})")
print(f"  lam_hat    = {np.exp(ll_ext):.4f}   (true: {lambda_true})")
print(f"  success    = {res_ext.success}")

**Observations:**
- The extended MLE recovers $\hat{\mu}_{\rm tot} \approx n_{\rm obs}$, as expected:
  the Poisson constraint acts as a self-normalisation.
- All other parameter estimates remain close to those from the standard unbinned MLE.
- The extended likelihood is particularly useful when the total yield itself carries
  information about the signal, e.g. in counting experiments or when the signal
  cross-section is the parameter of interest.